# Statistical Analysis of Self-Check Divergence

**Can shorter chain-of-thought reduce contradiction in model self-checks?**

This notebook demonstrates the evaluation pipeline from the artifact *Statistical Analysis of Self-Check Divergence*. It analyzes experiment results from GSM8K problems to test whether shorter chain-of-thought (CoT) reasoning reduces contradictions between model answers and their self-checks.

### Key findings from the full study (1,156 results):
- **Short CoT agreement rate**: 68.1% — **Medium**: 75.6% — **Long**: 54.8%
- **McNemar's test**: p = 0.0003 (short vs long agreement differs significantly)
- **Spearman trend**: ρ = -0.116, p = 0.0008 (longer CoT correlates with lower agreement)
- **Cohen's h**: 0.27 (medium effect size)
- **Divergence**: Long CoT shows positive divergence (agreement > accuracy), suggesting overconfidence

The demo uses a curated subset of 3 examples (one per CoT style) to illustrate the analysis pipeline.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2', 'tabulate==0.9.0')


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import json
import math
import re
import sys
from pathlib import Path
from typing import Any
from collections import defaultdict

import numpy as np
from scipy import stats
from loguru import logger
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# NumPy 2.0 compatibility shims (for older packages that may use deprecated APIs)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

# Configure logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

1

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-febf9b-longer-reasoning-chains-reduce-self/main/round-2/evaluation-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Metadata keys: {list(data['metadata'].keys())}")
print(f"Aggregated metrics: {list(data['metrics_agg'].keys())}")

Loaded data with 3 examples
Metadata keys: ['evaluation_name', 'description', 'hypothesis', 'experiment_model', 'temperature', 'bootstrap_iterations', 'ci_level', 'difficulty_validation', 'confidence_intervals', 'mcnemar_test', 'paired_sign_test', 'spearman_trend_test', 'divergence_analysis', 'effect_sizes', 'difficulty_stratified']
Aggregated metrics: ['total_results', 'short_agreement_rate', 'medium_agreement_rate', 'long_agreement_rate', 'short_accuracy', 'medium_accuracy', 'long_accuracy', 'mcnemar_p_value', 'sign_test_p_value', 'spearman_r', 'spearman_p_value', 'cohens_h', 'divergence_short', 'divergence_medium', 'divergence_long']


In [5]:
# ============================================================
# CONFIGURATION
# ============================================================
# Tunable parameters — set to minimum values for demo
# Original full-run values are commented for reference

BOOTSTRAP_ITERATIONS = 10  # Original: 1000
CI_LEVEL = 0.95            # Original: 0.95 (unchanged)

logger.info(f"Config: bootstrap_iterations={BOOTSTRAP_ITERATIONS}, ci_level={CI_LEVEL}")

20:26:51|INFO   |Config: bootstrap_iterations=10, ci_level=0.95


## Data Overview

The loaded data contains pre-computed evaluation results from the full experiment. Each example represents a GSM8K problem evaluated with a specific CoT style (short, medium, or long). The metadata contains all statistical test results from the full 1,156-result analysis.

In [6]:
# Display the examples in the demo dataset
examples = data['datasets'][0]['examples']

rows = []
for ex in examples:
    rows.append([
        ex['metadata_problem_index'],
        ex['metadata_cot_style'],
        ex['metadata_agreement'],
        ex['metadata_cot_correct'],
        ex['metadata_selfcheck_correct'],
        ex['eval_agreement_rate'],
        ex['eval_divergence']
    ])

print(tabulate(rows, headers=[
    'Problem', 'CoT Style', 'Agreement', 'CoT Correct', 'Self-Check Correct',
    'Agreement Rate', 'Divergence'
], tablefmt='grid'))

+-----------+-------------+-------------+---------------+----------------------+------------------+--------------+
|   Problem | CoT Style   | Agreement   | CoT Correct   | Self-Check Correct   |   Agreement Rate |   Divergence |
+===========+=============+=============+===============+======================+==================+==============+
|         0 | short       | True        | True          | True                 |         0.681347 |      -0.1503 |
+-----------+-------------+-------------+---------------+----------------------+------------------+--------------+
|         0 | medium      | True        | True          | True                 |         0.755844 |      -0.0675 |
+-----------+-------------+-------------+---------------+----------------------+------------------+--------------+
|         0 | long        | True        | True          | True                 |         0.548052 |       0.0494 |
+-----------+-------------+-------------+---------------+----------------------+

## Statistical Analyses

The following cells reproduce the key analysis functions from the original `eval.py` script. Since the demo data is a small subset, we use the pre-computed results from the full run (stored in `data['metadata']`) and demonstrate the analysis logic on the available examples.

In [7]:
# ============================================================
# ANALYSIS FUNCTIONS (from original eval.py)
# ============================================================

def compute_bootstrap_ci(agreement_rates: dict[str, float], n_per_style: dict[str, int]) -> dict:
    """Compute 95% confidence intervals via bootstrap resampling."""
    result = {}
    
    for style in ["short", "medium", "long"]:
        n = n_per_style.get(style, 0)
        if n == 0:
            result[style] = {"mean": 0.0, "ci_lower": 0.0, "ci_upper": 1.0}
            continue
        
        rate = agreement_rates[style]
        n_success = int(round(rate * n))
        
        # Bootstrap resampling
        resampled_means = []
        for _ in range(BOOTSTRAP_ITERATIONS):
            sample = np.random.binomial(1, rate, n)
            resampled_means.append(np.mean(sample))
        
        ci_lower = float(np.percentile(resampled_means, (1 - CI_LEVEL) / 2 * 100))
        ci_upper = float(np.percentile(resampled_means, (1 + CI_LEVEL) / 2 * 100))
        
        result[style] = {
            "mean": round(rate, 4),
            "ci_lower": round(ci_lower, 4),
            "ci_upper": round(ci_upper, 4),
            "n": n
        }
    
    return result


def compute_divergence(results: list[dict]) -> dict:
    """Compute divergence between accuracy and agreement rates."""
    by_style = defaultdict(lambda: {"acc": [], "agree": []})
    
    for r in results:
        style = r["cot_style"]
        by_style[style]["acc"].append(1 if r["cot_correct"] else 0)
        by_style[style]["agree"].append(1 if r["agreement"] else 0)
    
    divergence = {}
    for style in ["short", "medium", "long"]:
        if style in by_style:
            acc_rate = np.mean(by_style[style]["acc"])
            agree_rate = np.mean(by_style[style]["agree"])
            divergence[style] = {
                "accuracy": round(float(acc_rate), 4),
                "agreement": round(float(agree_rate), 4),
                "divergence": round(float(agree_rate - acc_rate), 4)
            }
        else:
            divergence[style] = {"accuracy": 0.0, "agreement": 0.0, "divergence": 0.0}
    
    return divergence


def compute_effect_sizes(mcnemar: dict, sign_test: dict, agreement_rates: dict[str, float]) -> dict:
    """Compute Cohen's h and odds ratio."""
    p1 = agreement_rates.get("short", 0)
    p2 = agreement_rates.get("long", 0)
    
    if p1 > 0 and p2 > 0:
        cohens_h = abs(2 * math.asin(math.sqrt(p1)) - 2 * math.asin(math.sqrt(p2)))
    else:
        cohens_h = 0.0
    
    discordant_b = mcnemar.get("discordant_pairs", {}).get("b", 0)
    discordant_c = mcnemar.get("discordant_pairs", {}).get("c", 0)
    
    if discordant_c > 0:
        odds_ratio = discordant_b / discordant_c
    else:
        odds_ratio = float('inf') if discordant_b > 0 else 1.0
    
    return {
        "cohens_h_short_vs_long": round(float(cohens_h), 4),
        "odds_ratio_mcnemar": round(float(odds_ratio), 4) if odds_ratio != float('inf') else "inf",
        "interpretation": "small" if cohens_h < 0.2 else ("medium" if cohens_h < 0.5 else "large")
    }

## Running Analyses on Demo Data

Now we reconstruct the per-example results from the demo data and run the analysis functions. We also compare against the pre-computed full-run results.

In [8]:
# Reconstruct results list from demo examples
results = []
for ex in examples:
    results.append({
        "problem_index": int(ex['metadata_problem_index']),
        "cot_style": ex['metadata_cot_style'],
        "agreement": ex['metadata_agreement'] == 'True',
        "cot_correct": ex['metadata_cot_correct'] == 'True',
        "selfcheck_correct": ex['metadata_selfcheck_correct'] == 'True',
    })

# Group by style
by_style = defaultdict(list)
for r in results:
    by_style[r["cot_style"]].append(r)

n_per_style = {style: len(group) for style, group in by_style.items()}
agreement_rates = {style: sum(1 for r in group if r["agreement"]) / len(group) 
                   for style, group in by_style.items()}

logger.info(f"Demo results by style: {n_per_style}")
logger.info(f"Demo agreement rates: {agreement_rates}")

# Run divergence analysis on demo data
demo_divergence = compute_divergence(results)
print("\nDemo Divergence Analysis:")
for style in ["short", "medium", "long"]:
    d = demo_divergence[style]
    print(f"  {style}: accuracy={d['accuracy']:.4f}, agreement={d['agreement']:.4f}, divergence={d['divergence']:.4f}")

# Run bootstrap CI on demo data (with reduced iterations)
demo_ci = compute_bootstrap_ci(agreement_rates, n_per_style)
print("\nDemo Bootstrap CI (" + str(BOOTSTRAP_ITERATIONS) + " iterations):")
for style in ["short", "medium", "long"]:
    ci = demo_ci[style]
    print(f"  {style}: mean={ci['mean']:.4f}, 95% CI=[{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]")

20:26:51|INFO   |Demo results by style: {'short': 1, 'medium': 1, 'long': 1}


20:26:51|INFO   |Demo agreement rates: {'short': 1.0, 'medium': 1.0, 'long': 1.0}



Demo Divergence Analysis:
  short: accuracy=1.0000, agreement=1.0000, divergence=0.0000
  medium: accuracy=1.0000, agreement=1.0000, divergence=0.0000
  long: accuracy=1.0000, agreement=1.0000, divergence=0.0000

Demo Bootstrap CI (10 iterations):
  short: mean=1.0000, 95% CI=[1.0000, 1.0000]
  medium: mean=1.0000, 95% CI=[1.0000, 1.0000]
  long: mean=1.0000, 95% CI=[1.0000, 1.0000]


## Full-Run Results (Pre-computed)

The metadata from the full 1,156-result run contains all statistical test results. Let's display the key findings.

In [9]:
metadata = data['metadata']
metrics = data['metrics_agg']

print("=" * 60)
print("FULL RUN SUMMARY (1,156 results)")
print("=" * 60)

# Agreement rates
print("\nAgreement Rates:")
print(f"  Short:  {metrics['short_agreement_rate']:.4f}")
print(f"  Medium: {metrics['medium_agreement_rate']:.4f}")
print(f"  Long:   {metrics['long_agreement_rate']:.4f}")

# Divergence
print("\nDivergence (Agreement - Accuracy):")
for style in ["short", "medium", "long"]:
    d = metadata['divergence_analysis'][style]
    print(f"  {style}: accuracy={d['accuracy']:.4f}, agreement={d['agreement']:.4f}, divergence={d['divergence']:.4f}")

# Statistical tests
print("\nStatistical Tests:")
print(f"  McNemar's test (short vs long): χ²={metadata['mcnemar_test']['chi2']:.4f}, p={metadata['mcnemar_test']['p_value']:.6f}")
print(f"  Paired sign test: z={metadata['paired_sign_test']['z']:.4f}, p={metadata['paired_sign_test']['p_value']:.6f}")
print(f"  Spearman trend: ρ={metadata['spearman_trend_test']['spearman_r']:.4f}, p={metadata['spearman_trend_test']['p_value']:.6f}")

# Effect sizes
es = metadata['effect_sizes']
print(f"\nEffect Sizes:")
print(f"  Cohen's h (short vs long): {es['cohens_h_short_vs_long']:.4f} ({es['interpretation']})")
print(f"  Odds ratio (McNemar): {es['odds_ratio_mcnemar']}")

FULL RUN SUMMARY (1,156 results)

Agreement Rates:
  Short:  0.6813
  Medium: 0.7558
  Long:   0.5481

Divergence (Agreement - Accuracy):
  short: accuracy=0.8316, agreement=0.6813, divergence=-0.1503
  medium: accuracy=0.8234, agreement=0.7558, divergence=-0.0675
  long: accuracy=0.4987, agreement=0.5481, divergence=0.0494

Statistical Tests:
  McNemar's test (short vs long): χ²=13.0381, p=0.000305
  Paired sign test: z=-3.6108, p=0.000391
  Spearman trend: ρ=-0.1161, p=0.000817

Effect Sizes:
  Cohen's h (short vs long): 0.2749 (medium)
  Odds ratio (McNemar): 2.0882


## Difficulty-Stratified Analysis

The full run also includes analysis stratified by problem difficulty (easy, medium, hard). This shows how CoT length effects vary across difficulty levels.

In [10]:
stratified = metadata['difficulty_stratified']

rows = []
for tier in ["easy", "medium", "hard"]:
    for style in ["short", "medium", "long"]:
        s = stratified[tier][style]
        rows.append([tier.capitalize(), style.capitalize(), s['n'], f"{s['agreement_rate']:.4f}", f"{s['accuracy']:.4f}"])

print(tabulate(rows, headers=['Tier', 'CoT Style', 'N', 'Agreement Rate', 'Accuracy'], tablefmt='grid'))

+--------+-------------+-----+------------------+------------+
| Tier   | CoT Style   |   N |   Agreement Rate |   Accuracy |
+========+=============+=====+==================+============+
| Easy   | Short       | 128 |           0.6953 |     0.8359 |
+--------+-------------+-----+------------------+------------+
| Easy   | Medium      | 128 |           0.7109 |     0.8125 |
+--------+-------------+-----+------------------+------------+
| Easy   | Long        | 128 |           0.4922 |     0.4766 |
+--------+-------------+-----+------------------+------------+
| Medium | Short       | 111 |           0.6667 |     0.8288 |
+--------+-------------+-----+------------------+------------+
| Medium | Medium      | 111 |           0.8018 |     0.8198 |
+--------+-------------+-----+------------------+------------+
| Medium | Long        | 111 |           0.6667 |     0.5586 |
+--------+-------------+-----+------------------+------------+
| Hard   | Short       | 147 |           0.6803 |     0

## Visualization

The final cell produces visualizations of the key findings: agreement rates across CoT styles, divergence patterns, and difficulty-stratified results.

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.set_style("whitegrid")

styles = ['short', 'medium', 'long']
style_labels = ['Short', 'Medium', 'Long']

# --- Panel 1: Agreement Rates with Confidence Intervals ---
ax1 = axes[0, 0]
ci_data = metadata['confidence_intervals']
means = [ci_data[s]['mean'] for s in styles]
ci_lo = [ci_data[s]['ci_lower'] for s in styles]
ci_hi = [ci_data[s]['ci_upper'] for s in styles]

bars = ax1.bar(style_labels, means, color=['#2196F3', '#4CAF50', '#F44336'], alpha=0.8, edgecolor='black')
ax1.errorbar(style_labels, means, yerr=[np.array(means) - np.array(ci_lo), np.array(ci_hi) - np.array(means)], 
             fmt='none', c='black', capsize=5, linewidth=1.5)
ax1.set_ylabel('Agreement Rate')
ax1.set_title('Self-Check Agreement Rate by CoT Length')
ax1.set_ylim(0.4, 0.85)
for bar, val in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)

# --- Panel 2: Divergence (Agreement - Accuracy) ---
ax2 = axes[0, 1]
div_data = metadata['divergence_analysis']
acc_rates = [div_data[s]['accuracy'] for s in styles]
agr_rates = [div_data[s]['agreement'] for s in styles]
div_rates = [div_data[s]['divergence'] for s in styles]

x = np.arange(len(styles))
width = 0.35
ax2.bar(x - width/2, acc_rates, width, label='Accuracy', color='#2196F3', alpha=0.8, edgecolor='black')
ax2.bar(x + width/2, agr_rates, width, label='Agreement', color='#FF9800', alpha=0.8, edgecolor='black')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax2.set_xticks(x)
ax2.set_xticklabels(style_labels)
ax2.set_ylabel('Rate')
ax2.set_title('Accuracy vs Agreement Rate')
ax2.legend()
for i, (a, g) in enumerate(zip(acc_rates, agr_rates)):
    ax2.text(i - width/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=8)
    ax2.text(i + width/2, g + 0.01, f'{g:.3f}', ha='center', fontsize=8)

# --- Panel 3: Difficulty-Stratified Agreement Rates ---
ax3 = axes[1, 0]
tiers = ['easy', 'medium', 'hard']
tier_labels = ['Easy', 'Medium', 'Hard']
colors = {'short': '#2196F3', 'medium': '#4CAF50', 'long': '#F44336'}

x = np.arange(len(tiers))
width = 0.25
for i, style in enumerate(styles):
    vals = [stratified[tier][style]['agreement_rate'] for tier in tiers]
    ax3.bar(x + i * width - width, vals, width, label=style.capitalize(), color=colors[style], alpha=0.8, edgecolor='black')

ax3.set_xticks(x)
ax3.set_xticklabels(tier_labels)
ax3.set_ylabel('Agreement Rate')
ax3.set_title('Agreement Rate by Difficulty & CoT Length')
ax3.legend(fontsize=8)
ax3.set_ylim(0.4, 0.85)

# --- Panel 4: Statistical Test Summary ---
ax4 = axes[1, 1]
ax4.axis('off')

summary_text = f"""STATISTICAL TEST RESULTS
{'='*40}

McNemar's Test (Short vs Long):
  χ² = {metadata['mcnemar_test']['chi2']:.4f}
  p = {metadata['mcnemar_test']['p_value']:.6f}
  Discordant: b={metadata['mcnemar_test']['discordant_pairs']['b']}, c={metadata['mcnemar_test']['discordant_pairs']['c']}

Paired Sign Test:
  z = {metadata['paired_sign_test']['z']:.4f}
  p = {metadata['paired_sign_test']['p_value']:.6f}
  Effective pairs: {metadata['paired_sign_test']['n_effective']}

Spearman Trend Test:
  ρ = {metadata['spearman_trend_test']['spearman_r']:.4f}
  p = {metadata['spearman_trend_test']['p_value']:.6f}

Effect Size:
  Cohen's h = {es['cohens_h_short_vs_long']:.4f} ({es['interpretation']})
  Odds ratio = {es['odds_ratio_mcnemar']}

CONCLUSION:
Longer CoT significantly reduces self-check
agreement (p < 0.001), suggesting increased
contradiction between model answers and
their self-evaluations."""

ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to 'results.png'")
print("\nKEY FINDING: Longer chain-of-thought reasoning leads to significantly")
print("lower self-check agreement rates, suggesting that extended reasoning")
print("increases the likelihood of contradictions between model answers")
print("and their self-evaluations.")


Visualization saved to 'results.png'

KEY FINDING: Longer chain-of-thought reasoning leads to significantly
lower self-check agreement rates, suggesting that extended reasoning
increases the likelihood of contradictions between model answers
and their self-evaluations.
